In [1]:
# 필요한 패키지
# pip install sqlalchemy pymysql pandas

import pandas as pd
from sqlalchemy import create_engine, text
from typing import Sequence, Optional

from datetime import datetime
import os
from ticker_list import ALL_TICKERS

# ── DB 접속 정보 ───────────────────────────────────────────────────────────────
from DATA.stock_invest_function import get_db_host

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TABLE_NAME = "Korea_company_valuation_ver2"  # 스키마: investar.TABLE_NAME

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# 1) forecast_date의 unique 값 추출
def get_unique_forecast_dates(include_null=False):
    q = f"""
        SELECT DISTINCT forecast_date
        FROM {TABLE_NAME}
        {"WHERE forecast_date IS NOT NULL" if not include_null else ""}
        ORDER BY forecast_date
    """
    with engine.begin() as conn:
        df = pd.read_sql(q, conn, parse_dates=["forecast_date"])
    return df["forecast_date"]

# 2) (ticker, forecast_date, keyword)로 indicator에 keyword가 포함된 값 조회 + date 기준 정렬
#    여러 indicator가 매칭되면 행으로 반환(롱 포맷). wide=True면 indicator별 칼럼으로 피벗.
def get_series_by_keyword(ticker, forecast_date, keyword, wide=False):
    sql = text(f"""
        SELECT `date`, `ticker`, `indicator`, `value`, `forecast_date`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND forecast_date = :fdate
          AND indicator LIKE :kw
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={"ticker": ticker, "fdate": forecast_date, "kw": f"%{keyword}%"},
            parse_dates=["date", "forecast_date"]
        )
    # 숫자형 보정
    if not df.empty:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if wide and not df.empty:
        df_wide = df.pivot_table(index="date", columns="indicator", values="value", aggfunc="last").sort_index()
        df_wide = df_wide.rename_axis(None, axis=1)
        return df_wide
    return df  # 롱 포맷: date, indicator, value …

# 3) indicator의 unique 값 추출
def get_unique_indicators(keyword=None):
    cond = "" if not keyword else "WHERE indicator LIKE :kw"
    sql = text(f"SELECT DISTINCT indicator FROM {TABLE_NAME} {cond} ORDER BY indicator")
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params=(None if not keyword else {"kw": f"%{keyword}%"}))
    return df["indicator"]

# 4) (ticker, indicator, forecast_date 두 개) 입력 시 두 기간 차이 비교
#    반환: date 기준 병합(outer), col: value_fd1, value_fd2, diff = fd2 - fd1
def compare_indicator_between_dates(ticker, indicator, forecast_date_1, forecast_date_2):
    base_sql = text(f"""
        SELECT `date`, `value`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND indicator = :indicator
          AND forecast_date = :fdate
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df1 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_1},
            parse_dates=["date"]
        )
        df2 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_2},
            parse_dates=["date"]
        )

    # 숫자형 보정
    for d in (df1, df2):
        if not d.empty:
            d["value"] = pd.to_numeric(d["value"], errors="coerce")

    df1 = df1.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_1).date()}"})
    df2 = df2.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_2).date()}"})

    out = pd.merge(df1, df2, on="date", how="outer").sort_values("date").set_index("date")
    if out.shape[1] == 2:
        cols = out.columns.tolist()
        out["diff"] = out[cols[1]] - out[cols[0]]  # fd2 - fd1
    return out

# -*- coding: utf-8 -*-
from __future__ import annotations
import pandas as pd
from typing import Optional, Union, Sequence, Tuple
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────────────────────────────────────────

def make_engine(db_info: dict):
    url = (
        "mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
        "?charset=utf8mb4"
    ).format(**db_info)
    return create_engine(url, pool_recycle=3600, pool_pre_ping=True)


# ─────────────────────────────────────────────────────────────────────────────
# 1) 특정 forecast_date로 유니크 티커 조회
#   - forecast_date가 None → forecast_date IS NULL 조건
#   - forecast_date가 'YYYY-MM-DD' 나 'YYYY-MM-DD HH:MM:SS' → 해당 일자/시각 매칭
#   - '같은 날'로 묶고 싶다면 use_date_only=True 로 DATE(forecast_date)=DATE(:dt)
# ─────────────────────────────────────────────────────────────────────────────
def get_unique_tickers_by_forecast_date(
    db_info: dict,
    forecast_date: Optional[str] = None,
    table_name: str = "valuation_forecast_result",
    use_date_only: bool = True,
) -> pd.Series:
    """
    Return: pd.Series of unique tickers (name='ticker')
    """
    engine = make_engine(db_info)
    with engine.connect() as conn:
        if forecast_date is None:
            sql = text(f"""
                SELECT DISTINCT ticker
                FROM {table_name}
                WHERE forecast_date IS NULL
                ORDER BY ticker
            """)
            df = pd.read_sql(sql, conn)
        else:
            if use_date_only:
                sql = text(f"""
                    SELECT DISTINCT ticker
                    FROM {table_name}
                    WHERE DATE(forecast_date) = DATE(:dt)
                    ORDER BY ticker
                """)
            else:
                sql = text(f"""
                    SELECT DISTINCT ticker
                    FROM {table_name}
                    WHERE forecast_date = :dt
                    ORDER BY ticker
                """)
            df = pd.read_sql(sql, conn, params={"dt": forecast_date})

    return df["ticker"]


# ─────────────────────────────────────────────────────────────────────────────
# 2) indicator별 date1→date2 변화율 계산
#   - indicator: str 또는 [str, ...]  (None/'all'은 전체)
#   - 변화율 = (v2 / v1 - 1).  v1=0 또는 결측은 안전하게 제외
#   - 결과 정렬: pct_change(%) 내림차순
# Columns:
#   ['indicator','ticker','date1','date2','value_date1','value_date2',
#    'abs_change','pct_change']
# ─────────────────────────────────────────────────────────────────────────────

def get_indicator_change_rates(
    db_info: dict,
    forecast_date: Optional[str],
    indicator: Optional[Union[str, Sequence[str]]] = None,
    date1: str = "2027-03-31",
    date2: str = "2027-06-30",
    table_name: str = "valuation_forecast_result",
    use_date_only_for_forecast: bool = True,
    drop_zero_base: bool = True,
    tolerance_days: int = 3,
) -> pd.DataFrame:
    """
    안정화 버전: date1/date2 근사 매칭 + KeyError 방지
    """
    engine = make_engine(db_info)
    with engine.connect() as conn:
        conds, params = [], {}

        # forecast_date filter (SQL)
        if forecast_date is None:
            conds.append("forecast_date IS NULL")
        else:
            if use_date_only_for_forecast:
                conds.append("DATE(forecast_date) = DATE(:fdt)")
            else:
                conds.append("forecast_date = :fdt")
            params["fdt"] = forecast_date

        # indicator filter
        if indicator is None or (isinstance(indicator, str) and indicator.lower() == "all"):
            pass
        else:
            if isinstance(indicator, str):
                indicator_list = [indicator]
            else:
                indicator_list = list(indicator)
            placeholders = []
            for i, it in enumerate(indicator_list):
                key = f"i{i}"
                params[key] = it
                placeholders.append(f":{key}")
            conds.append(f"indicator IN ({', '.join(placeholders)})")

        where_sql = " AND ".join(conds) if conds else "1=1"

        # ---- SQL (DATE 변환 제거) ----
        sql = text(f"""
            SELECT
                `date` AS d,
                `ticker`,
                `indicator`,
                CAST(REPLACE(`value`, ',', '') AS DECIMAL(38,8)) AS value
            FROM {table_name}
            WHERE {where_sql}
              AND `date` IS NOT NULL
        """)
        raw = pd.read_sql(sql, conn, params=params, parse_dates=["d"])

    if raw.empty:
        print("⚠️ DB에서 forecast_date 조건에 맞는 데이터가 없습니다.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    # --- 전처리 ---
    raw["value"] = pd.to_numeric(raw["value"], errors="coerce")
    raw = raw.dropna(subset=["value"])
    if raw.empty:
        print("⚠️ value가 모두 결측이거나 숫자형 변환 실패.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    # 날짜 정규화
    raw["d"] = pd.to_datetime(raw["d"]).dt.floor("D")
    d1 = pd.to_datetime(date1)
    d2 = pd.to_datetime(date2)

    def near_mask(series, target, tol):
        delta = (series - target).abs().dt.days
        return delta <= tol

    mask = near_mask(raw["d"], d1, tolerance_days) | near_mask(raw["d"], d2, tolerance_days)
    raw2 = raw.loc[mask].copy()

    if raw2.empty:
        print(f"⚠️ date1={date1}, date2={date2} ±{tolerance_days}일 내 데이터 없음.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    # ---- pivot ----
    piv = (
        raw2
        .pivot_table(index=["indicator","ticker"], columns="d", values="value", aggfunc="last")
        .reset_index()
    )

    if piv.empty:
        print("⚠️ pivot 결과가 비었습니다.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    print(f"✅ pivot 컬럼 목록:\n{list(piv.columns)}")  # 디버그용

    # ---- 날짜 컬럼 실제 존재 확인 ----
    date_cols = [c for c in piv.columns if isinstance(c, pd.Timestamp)]
    if not date_cols:
        print("⚠️ pivot에 날짜 컬럼이 없습니다.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    def get_closest_col(df_cols, target):
        deltas = [abs((c - target).days) for c in df_cols]
        return df_cols[deltas.index(min(deltas))]

    d1_actual = get_closest_col(date_cols, d1)
    d2_actual = get_closest_col(date_cols, d2)
    print(f"📅 실제 사용 날짜: d1_actual={d1_actual.date()}, d2_actual={d2_actual.date()}")

    # ---- rename 안전 처리 ----
    rename_map = {}
    if d1_actual in piv.columns:
        rename_map[d1_actual] = "value_date1"
    if d2_actual in piv.columns:
        rename_map[d2_actual] = "value_date2"

    piv = piv.rename(columns=rename_map)

    # ✅ rename 후에도 컬럼이 없으면 생성
    if "value_date1" not in piv.columns:
        piv["value_date1"] = pd.NA
    if "value_date2" not in piv.columns:
        piv["value_date2"] = pd.NA

    out = piv[["indicator","ticker","value_date1","value_date2"]].copy()

    # 결측/0 제거
    out = out.dropna(subset=["value_date1","value_date2"])
    if drop_zero_base:
        out = out[out["value_date1"] != 0]

    if out.empty:
        print("⚠️ 유효한 변화 계산 대상이 없습니다.")
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2","value_date1","value_date2","abs_change","pct_change"
        ])

    # ---- 변화율 계산 ----
    out["abs_change"] = out["value_date2"] - out["value_date1"]
    out["pct_change"] = (out["value_date2"] / out["value_date1"] - 1.0) * 100.0
    out.insert(2, "date1", d1_actual.date())
    out.insert(3, "date2", d2_actual.date())

    out = out.sort_values(["indicator","pct_change"], ascending=[True, False]).reset_index(drop=True)
    print(f"✅ 최종 결과 행 수: {len(out)}")
    return out


CHUNKSIZE  = 200_000

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# ── 0) 로우 카운트 참고용 (옵션) ─────────────────────────────────────────────
with engine.begin() as conn:
    rowcount = pd.read_sql(
        text(f"SELECT COUNT(*) AS c FROM {TABLE_NAME}"), conn
    )["c"].iloc[0]
print(f"[INFO] total rows in {TABLE_NAME}: {rowcount:,}")

# ── 공통 SELECT (날짜 컬럼을 함께 들고 오기) ─────────────────────────────────
# 필요한 경우 * 로 바꾸세요. 아래는 가장 많이 쓰는 컬럼 예시입니다.
BASE_SQL = f"""
SELECT
  `date`,
  `ticker`,
  `indicator`,
  `value`,
  `forecast_date`
FROM {TABLE_NAME}
"""

# ============ (A) 메모리가 충분한 경우: 단일 DataFrame 로드 ==================
def load_full_table_as_dataframe(chunksize=CHUNKSIZE) -> pd.DataFrame:
    """
    테이블 전체를 청크로 읽어 합친 DataFrame 반환.
    메모리가 충분할 때만 사용하세요.
    """
    all_parts = []
    total = 0
    with engine.begin() as conn:
        for i, chunk in enumerate(
            pd.read_sql(
                text(BASE_SQL), conn,
                chunksize=chunksize,
                parse_dates=["date", "forecast_date"],  # 날짜 컬럼 파싱
            ),
            start=1
        ):
            total += len(chunk)
            all_parts.append(chunk)
            print(f"[CHUNK {i}] rows={len(chunk):,}  |  accumulated={total:,}")

    if not all_parts:
        print("[WARN] No rows read.")
        return pd.DataFrame(columns=["date","ticker","indicator","value","forecast_date"])

    df_all = pd.concat(all_parts, axis=0, ignore_index=True)
    print(f"[DONE] Loaded DataFrame with rows={len(df_all):,}")
    return df_all


# 필요시 교체
def make_engine(db):
    url = f"mysql+pymysql://{db['user']}:{db['password']}@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    return create_engine(url, pool_pre_ping=True, future=True)

def get_last_market_cap(
    db_info: dict,
    table_name: str,
    ticker_list: Sequence[str],
    indicator_exact: Optional[str] = None,     # 예: '시가총액' 또는 'market_cap'
    indicator_like: Optional[str] = None,      # 예: '%시가총액%' 또는 'market_cap%'
) -> pd.DataFrame:
    """
    입력 티커들에 대해 '마지막 날짜'의 시가총액만 추출 (롱 포맷)
    반환 칼럼: ['ticker','date','last_market_cap']
    """

    if not ticker_list:
        return pd.DataFrame(columns=["ticker","date","last_market_cap"])

    engine = make_engine(db_info)
    params = {"tickers": tuple(ticker_list)}

    # indicator 조건
    ind_cond = "1=1"
    if indicator_exact:
        ind_cond = "indicator = :ind_exact"
        params["ind_exact"] = indicator_exact
    elif indicator_like:
        ind_cond = "indicator LIKE :ind_like"
        params["ind_like"] = indicator_like
    else:
        # 기본값: 한국어/영문 둘 다 시도
        ind_cond = "(indicator = '시가총액' OR indicator LIKE 'market_cap%')"

    # 1) 우선 윈도 함수(ROW_NUMBER) 사용 (MySQL 8+/MariaDB 10.2+)
    sql_win = text(f"""
        WITH base AS (
            SELECT
                `date`,
                `ticker`,
                CAST(REPLACE(`value`, ',', '') AS DECIMAL(38,8)) AS value
            FROM {table_name}
            WHERE ticker IN :tickers
              AND {ind_cond}
              AND `date` IS NOT NULL
        ),
        ranked AS (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY `date` DESC) AS rn
            FROM base
        )
        SELECT
            ticker,
            DATE(`date`) AS `date`,
            value AS last_market_cap
        FROM ranked
        WHERE rn = 1
        ORDER BY ticker
    """)

    try:
        with engine.begin() as conn:
            df = pd.read_sql(sql_win, conn, params=params, parse_dates=["date"])
        if not df.empty:
            return df[["ticker","date","last_market_cap"]]
    except Exception:
        pass  # 구버전 DB면 아래 조인 방식으로 폴백

    # 2) 폴백: 윈도 함수 없이 MAX(date) 조인
    sql_join = text(f"""
        SELECT b.ticker,
               DATE(b.`date`) AS `date`,
               b.value AS last_market_cap
        FROM (
            SELECT ticker, MAX(`date`) AS max_date
            FROM {table_name}
            WHERE ticker IN :tickers
              AND {ind_cond}
              AND `date` IS NOT NULL
            GROUP BY ticker
        ) m
        JOIN (
            SELECT
                `date`, `ticker`,
                CAST(REPLACE(`value`, ',', '') AS DECIMAL(38,8)) AS value
            FROM {table_name}
            WHERE ticker IN :tickers
              AND {ind_cond}
              AND `date` IS NOT NULL
        ) b
          ON b.ticker = m.ticker AND b.`date` = m.max_date
        ORDER BY b.ticker
    """)

    with engine.begin() as conn:
        df2 = pd.read_sql(sql_join, conn, params=params, parse_dates=["date"])
    return df2[["ticker","date","last_market_cap"]]

# ── 사용 예시 ─────────────────────────────────────────
# db_info = { 'host': ..., 'port': 3307, 'user': 'stox7412', 'password': '...', 'database': 'investar' }
# ticker_list = ["A005930","A000660","A035420"]
# df_last = get_last_market_cap(db_info, "Korea_company_valuation_ver2", ticker_list,
#                               indicator_exact="시가총액")  # 또는 indicator_like="%market_cap%"
# print(df_last)


[INFO] total rows in Korea_company_valuation_ver2: 411,490


In [2]:
print(get_unique_forecast_dates().tail())

4   2025-11-09
5   2025-11-12
6   2025-11-18
7   2025-11-24
8   2025-11-25
Name: forecast_date, dtype: datetime64[ns]


In [7]:
from datetime import datetime
import os
from ticker_list import ALL_TICKERS

# 오늘 날짜
today_date = datetime.now().strftime("%Y%m%d")

# 예측 날짜 설정 (필요시 수정)
FORECAST_DATE = "2025-11-24"

# 저장 경로
output_dir = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Results\Korea\Revenue"
output_filename = f"revenue_forecast_total_{today_date}.xlsx"
output_path = os.path.join(output_dir, output_filename)

# 결과를 저장할 빈 리스트
all_results = []

print("=" * 70)
print(f"Revenue 예측 데이터 수집 시작")
print(f"예측 날짜: {FORECAST_DATE}")
print(f"전체 티커 수: {len(ALL_TICKERS)}")
print("=" * 70)

# 티커별로 반복
for idx, ticker_info in enumerate(ALL_TICKERS, 1):
    ticker = ticker_info['ticker']

    print(f"\n[{idx}/{len(ALL_TICKERS)}] 처리 중: {ticker}")

    try:
        # get_series_by_keyword 함수 호출 (이 함수는 이미 정의되어 있다고 가정)
        ex_long = get_series_by_keyword(
            ticker=ticker,
            forecast_date=FORECAST_DATE,
            keyword="rev",
            wide=False
        )

        # 데이터가 없는 경우 건너뛰기
        if ex_long is None or len(ex_long) == 0:
            print(f"  ⚠️  데이터 없음 - 건너뜀")
            continue

        # ex_long을 pivot 테이블로 변환
        ex_pivot = (
            ex_long
            .pivot_table(
                index=["date", "ticker"],
                columns="indicator",
                values="value",
                aggfunc="last"
            )
            .reset_index()
        )

        # 컬럼명 정리
        ex_pivot.columns.name = None

        # 필요한 컬럼만 추출
        available_columns = ['date', 'ticker']
        forecast_columns = ['revenue_ets', 'revenue_prophet', 'revenue_sarima',
                          'revenue_sarima_exog', 'revenue_theta']

        # 실제 존재하는 컬럼만 선택
        for col in forecast_columns:
            if col in ex_pivot.columns:
                available_columns.append(col)

        extracted_df = ex_pivot[available_columns].copy()
        extracted_df = extracted_df.tail(12)

        # 결과 리스트에 추가
        all_results.append(extracted_df)

        print(f"  ✓ 성공: {len(extracted_df)} 행 수집")

    except Exception as e:
        print(f"  ✗ 오류 발생: {str(e)}")
        continue

print("\n" + "=" * 70)
print("데이터 수집 완료")
print("=" * 70)

# 모든 결과를 하나의 DataFrame으로 결합
if len(all_results) > 0:
    final_df = pd.concat(all_results, ignore_index=True)

    print(f"\n최종 데이터프레임 크기: {final_df.shape}")
    print(f"  - 총 행 수: {len(final_df):,}")
    print(f"  - 총 열 수: {len(final_df.columns)}")
    print(f"  - 티커 수: {final_df['ticker'].nunique()}")

    # 디렉토리가 없으면 생성
    os.makedirs(output_dir, exist_ok=True)

    # Excel 파일로 저장
    print(f"\n파일 저장 중...")
    final_df.to_excel(output_path, index=False, engine='openpyxl')

    print(f"✓ 저장 완료: {output_path}")

    # 미리보기
    print("\n데이터 미리보기 (처음 5행):")
    print(final_df.head())

else:
    print("\n⚠️  수집된 데이터가 없습니다.")

print("\n" + "=" * 70)
print("작업 완료")
print("=" * 70)


# TICKER = "A005930"
# FORECAST_DATE = "2025-11-09"
#
# ex_long = get_series_by_keyword(ticker=TICKER , forecast_date=FORECAST_DATE, keyword="rev", wide=False)
#
# # ex_long → indicator를 컬럼으로 피벗
# ex_pivot = (
#     ex_long
#     .pivot_table(
#         index=["date", "ticker"],      # 행 인덱스
#         columns="indicator",           # 열로 변환할 컬럼
#         values="value",                # 값으로 쓸 컬럼
#         aggfunc="last"                 # 중복 시 마지막 값 사용
#     )
#     .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
# )
#
# # 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
# ex_pivot.columns.name = None
#
# extracted_df = ex_pivot[['date', 'revenue_ets', 'revenue_prophet', 'revenue_sarima', 'revenue_sarima_exog', 'revenue_theta']]
# extracted_df['ticker'] = TICKER

Revenue 예측 데이터 수집 시작
예측 날짜: 2025-11-24
전체 티커 수: 54

[1/54] 처리 중: A005930
  ✓ 성공: 12 행 수집

[2/54] 처리 중: A000660
  ✓ 성공: 12 행 수집

[3/54] 처리 중: A051910
  ✓ 성공: 12 행 수집

[4/54] 처리 중: A035420
  ✓ 성공: 12 행 수집

[5/54] 처리 중: A005380
  ✓ 성공: 12 행 수집

[6/54] 처리 중: A006400
  ✓ 성공: 12 행 수집

[7/54] 처리 중: A035720
  ✓ 성공: 12 행 수집

[8/54] 처리 중: A000270
  ✓ 성공: 12 행 수집

[9/54] 처리 중: A068270
  ✓ 성공: 12 행 수집

[10/54] 처리 중: A207940
  ✓ 성공: 12 행 수집

[11/54] 처리 중: A042700
  ✓ 성공: 12 행 수집

[12/54] 처리 중: A043150
  ✓ 성공: 12 행 수집

[13/54] 처리 중: A131290
  ✓ 성공: 12 행 수집

[14/54] 처리 중: A006910
  ✓ 성공: 12 행 수집

[15/54] 처리 중: A140860
  ✓ 성공: 12 행 수집

[16/54] 처리 중: A009150
  ✓ 성공: 12 행 수집

[17/54] 처리 중: A095610
  ✓ 성공: 12 행 수집

[18/54] 처리 중: A001440
  ✓ 성공: 12 행 수집

[19/54] 처리 중: A000500
  ⚠️  데이터 없음 - 건너뜀

[20/54] 처리 중: A004000
  ⚠️  데이터 없음 - 건너뜀

[21/54] 처리 중: A010120
  ⚠️  데이터 없음 - 건너뜀

[22/54] 처리 중: A044820
  ⚠️  데이터 없음 - 건너뜀

[23/54] 처리 중: A010140
  ⚠️  데이터 없음 - 건너뜀

[24/54] 처리 중: A011780
  ⚠️  데이터 없음 - 건너뜀

[25

In [8]:
ex_pivot[['date', 'revenue_ets', 'revenue_prophet', 'revenue_sarima', 'revenue_sarima_exog', 'revenue_theta']].tail(10)
ex_pivot

,date,ticker,revenue_ets,revenue_ets_ttm,revenue_lstm,revenue_lstm_ttm,revenue_prophet,revenue_prophet_ttm,revenue_sarima,revenue_sarima_exog,revenue_sarima_exog_ttm,revenue_sarima_ttm,revenue_theta,revenue_theta_ttm
0,2004-12-31,A068270,0.000000e+00,NaN,0.0,NaN,0.000000e+00,NaN,0.000000e+00,0.000000e+00,NaN,NaN,0.000000e+00,NaN
1,2005-03-31,A068270,7.854353e+06,NaN,7854353.0,NaN,7.854353e+06,NaN,7.854353e+06,7.854353e+06,NaN,NaN,7.854353e+06,NaN
2,2005-06-30,A068270,8.448858e+06,NaN,8448858.0,NaN,8.448858e+06,NaN,8.448858e+06,8.448858e+06,NaN,NaN,8.448858e+06,NaN
3,2005-09-30,A068270,9.200098e+06,2.550331e+07,9200098.0,2.550331e+07,9.200098e+06,2.550331e+07,9.200098e+06,9.200098e+06,2.550331e+07,2.550331e+07,9.200098e+06,2.550331e+07
4,2005-12-31,A068270,1.036475e+07,3.586806e+07,10364747.0,3.586806e+07,1.036475e+07,3.586806e+07,1.036475e+07,1.036475e+07,3.586806e+07,3.586806e+07,1.036475e+07,3.586806e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,2025-12-31,A068270,9.777281e+08,3.729222e+09,810711872.0,3.399141e+09,8.486689e+08,3.481787e+09,1.057310e+09,9.587082e+08,3.721860e+09,3.854281e+09,9.858506e+08,3.763830e+09
85,2026-03-31,A068270,9.537309e+08,3.841041e+09,842394432.0,3.399624e+09,8.671776e+08,3.507053e+09,1.098576e+09,1.072331e+09,3.952279e+09,4.110945e+09,9.158561e+08,3.837774e+09
86,2026-06-30,A068270,9.951121e+08,3.874696e+09,866971008.0,3.305137e+09,8.858920e+08,3.431487e+09,1.152652e+09,1.090290e+09,4.081112e+09,4.302140e+09,9.941881e+08,3.870505e+09
87,2026-09-30,A068270,9.975571e+08,3.924128e+09,895525248.0,3.415603e+09,9.048121e+08,3.506551e+09,1.183938e+09,NaN,NaN,4.492475e+09,1.013103e+09,3.908998e+09


In [23]:
ex_pivot.tail(10)

,date,ticker,revenue_ets,revenue_ets_ttm,revenue_lstm,revenue_lstm_ttm,revenue_prophet,revenue_prophet_ttm,revenue_sarima,revenue_sarima_exog,revenue_sarima_exog_ttm,revenue_sarima_ttm,revenue_theta,revenue_theta_ttm
82,2024-09-30,A009160,1.885378e+08,7.102140e+08,1.885378e+08,7.102140e+08,1.885378e+08,7.102140e+08,1.885378e+08,1.885378e+08,7.102140e+08,7.102140e+08,1.885378e+08,7.102140e+08
83,2024-12-31,A009160,2.471866e+08,7.850618e+08,2.471866e+08,7.850618e+08,2.471866e+08,7.850618e+08,2.471866e+08,2.471866e+08,7.850618e+08,7.850618e+08,2.471866e+08,7.850618e+08
84,2025-03-31,A009160,2.302432e+08,8.792033e+08,2.302432e+08,8.792033e+08,2.302432e+08,8.792033e+08,2.302432e+08,2.302432e+08,8.792033e+08,8.792033e+08,2.302432e+08,8.792033e+08
85,2025-06-30,A009160,3.014763e+08,9.674439e+08,3.014763e+08,9.674439e+08,3.014763e+08,9.674439e+08,3.014763e+08,3.014763e+08,9.674439e+08,9.674439e+08,3.014763e+08,9.674439e+08
86,2025-09-30,A009160,2.770239e+08,1.055930e+09,1.952366e+08,9.741427e+08,2.141455e+08,9.930515e+08,2.836266e+08,NaN,NaN,1.062533e+09,2.678927e+08,1.046799e+09
87,2025-12-31,A009160,3.137294e+08,1.122473e+09,2.033318e+08,9.302879e+08,2.185124e+08,9.643773e+08,3.286396e+08,NaN,NaN,1.143986e+09,3.162471e+08,1.115859e+09
88,2026-03-31,A009160,2.947903e+08,1.187020e+09,2.101862e+08,9.102309e+08,2.227845e+08,9.569186e+08,3.052787e+08,NaN,NaN,1.219021e+09,2.763118e+08,1.161928e+09
89,2026-06-30,A009160,3.299119e+08,1.215456e+09,2.184492e+08,8.272039e+08,2.271040e+08,8.825463e+08,3.626327e+08,NaN,NaN,1.280178e+09,3.104684e+08,1.170920e+09
90,2026-09-30,A009160,3.110612e+08,1.249493e+09,2.233950e+08,8.553622e+08,2.314710e+08,8.998718e+08,3.419777e+08,NaN,NaN,1.338529e+09,2.758308e+08,1.178858e+09
91,2026-12-31,A009160,3.477667e+08,1.283530e+09,2.297007e+08,8.817311e+08,2.358379e+08,9.171973e+08,3.846560e+08,NaN,NaN,1.394545e+09,3.255490e+08,1.188160e+09


In [9]:
TICKER = 'A000660'

psr_long = get_series_by_keyword(ticker=TICKER, forecast_date= FORECAST_DATE, keyword="psr", wide=False)

# ex_long → indicator를 컬럼으로 피벗
psr_pivot = (
    psr_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
psr_pivot.columns.name = None

# 확인
print(psr_pivot.head())

        date   ticker       psr  psr_ETS  psr_LSTM  psr_Prophet  \
0 2015-05-31  A000660  2.605408      NaN       NaN          NaN   
1 2015-06-30  A000660  2.156728      NaN       NaN          NaN   
2 2015-07-31  A000660  1.891599      NaN       NaN          NaN   
3 2015-08-31  A000660  1.784510      NaN       NaN          NaN   
4 2015-09-30  A000660  1.672355      NaN       NaN          NaN   

   psr_SARIMA_exog  psr_SARIMA_noexog  psr_Theta  
0              NaN                NaN        NaN  
1              NaN                NaN        NaN  
2              NaN                NaN        NaN  
3              NaN                NaN        NaN  
4              NaN                NaN        NaN  


In [12]:
psr_pivot.tail(24)

,date,ticker,psr,psr_ETS,psr_LSTM,psr_Prophet,psr_SARIMA_exog,psr_SARIMA_noexog,psr_Theta
127,2025-12-31,A000660,NaN,6.524068,6.244461,3.757825,6.497100,6.122831,6.367536
128,2026-01-31,A000660,NaN,6.611796,6.760679,3.874141,6.603580,6.726673,6.375044
129,2026-02-28,A000660,NaN,6.436870,7.156620,3.631560,6.375858,6.564157,6.382552
130,2026-03-31,A000660,NaN,6.485794,7.457555,3.702849,6.475194,6.095701,6.390060
131,2026-04-30,A000660,NaN,6.463014,7.661674,3.639071,6.382959,6.365377,6.397568
132,2026-05-31,A000660,NaN,6.524450,7.725162,3.825497,6.522517,6.824823,6.405076
133,2026-06-30,A000660,NaN,6.736926,7.589467,4.189515,7.023955,6.702400,6.412584
134,2026-07-31,A000660,NaN,6.617114,7.238781,3.984377,6.802049,6.612315,6.420092
135,2026-08-31,A000660,NaN,6.444629,6.573061,3.711183,6.519048,6.853707,6.427600
136,2026-09-30,A000660,NaN,6.558679,5.523386,3.879290,6.723561,6.531880,6.435108


In [13]:
mc_long = get_series_by_keyword(ticker=TICKER, forecast_date=FORECAST_DATE, keyword="mc_", wide=False)

# ex_long → indicator를 컬럼으로 피벗
mc_pivot = (
    mc_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
mc_pivot.columns.name = None

# 확인
print(mc_pivot.head())

        date   ticker        mc_ets       mc_lstm    mc_prophet  \
0 2025-12-31  A000660  5.672864e+11  4.175307e+11  2.660536e+11   
1 2026-01-31  A000660  5.749147e+11  4.520472e+11  2.742888e+11   
2 2026-02-28  A000660  5.597043e+11  4.785215e+11  2.571141e+11   
3 2026-03-31  A000660  6.007753e+11  4.747805e+11  2.556458e+11   
4 2026-04-30  A000660  5.986652e+11  4.877756e+11  2.512426e+11   

   mc_sarima_exog  mc_sarima_noexog      mc_theta  
0    5.669274e+11      5.087851e+11  5.396603e+11  
1    5.762188e+11      5.589623e+11  5.402967e+11  
2    5.563481e+11      5.454577e+11  5.409330e+11  
3    6.033319e+11      5.067935e+11  5.735853e+11  
4    5.947379e+11      5.292143e+11  5.742592e+11  


In [14]:
mc_pivot.tail(25)

,date,ticker,mc_ets,mc_lstm,mc_prophet,mc_sarima_exog,mc_sarima_noexog,mc_theta
0,2025-12-31,A000660,5.672864e+11,4.175307e+11,2.660536e+11,5.669274e+11,5.087851e+11,5.396603e+11
1,2026-01-31,A000660,5.749147e+11,4.520472e+11,2.742888e+11,5.762188e+11,5.589623e+11,5.402967e+11
2,2026-02-28,A000660,5.597043e+11,4.785215e+11,2.571141e+11,5.563481e+11,5.454577e+11,5.409330e+11
3,2026-03-31,A000660,6.007753e+11,4.747805e+11,2.556458e+11,6.033319e+11,5.067935e+11,5.735853e+11
4,2026-04-30,A000660,5.986652e+11,4.877756e+11,2.512426e+11,5.947379e+11,5.292143e+11,5.742592e+11
5,2026-05-31,A000660,6.043560e+11,4.918175e+11,2.641135e+11,6.077413e+11,5.674124e+11,5.749332e+11
6,2026-06-30,A000660,6.373075e+11,4.297266e+11,2.637957e+11,7.002137e+11,5.318500e+11,5.791779e+11
7,2026-07-31,A000660,6.259733e+11,4.098703e+11,2.508790e+11,6.780921e+11,5.247016e+11,5.798560e+11
8,2026-08-31,A000660,6.096564e+11,3.721762e+11,2.336771e+11,6.498799e+11,5.438566e+11,5.805341e+11
9,2026-09-30,A000660,6.334859e+11,3.268576e+11,2.485844e+11,7.097594e+11,4.819812e+11,5.847997e+11


In [15]:
# ── 2️⃣ 함수 실행 ───────────────────────────────────────────
df_change = get_indicator_change_rates(
    db_info=db_info,
    forecast_date="2025-11-24",            # 📌 이 forecast_date와 일치하는 데이터 전체를 SQL에서 추출
    indicator="mc_sarima_noexog",            # 또는 None → 전체 indicator
    date1="2025-12-31",                    # 📈 비교 시작일
    date2="2026-12-31",                    # 📉 비교 종료일
    table_name="Korea_company_valuation_ver2",  # 실제 테이블명
    use_date_only_for_forecast=True,       # 날짜만 비교 (시각 무시)
    drop_zero_base=True                    # 0으로 나누는 행 제거
)

print(df_change.head())

✅ pivot 컬럼 목록:
['indicator', 'ticker', Timestamp('2025-12-31 00:00:00'), Timestamp('2026-12-31 00:00:00')]
📅 실제 사용 날짜: d1_actual=2025-12-31, d2_actual=2026-12-31
✅ 최종 결과 행 수: 17
d         indicator   ticker       date1       date2   value_date1  \
0  mc_sarima_noexog  A006910  2025-12-31  2026-12-31  4.388543e+08   
1  mc_sarima_noexog  A131290  2025-12-31  2026-12-31  8.282508e+08   
2  mc_sarima_noexog  A140860  2025-12-31  2026-12-31  2.673950e+09   
3  mc_sarima_noexog  A009150  2025-12-31  2026-12-31  2.281641e+10   
4  mc_sarima_noexog  A001440  2025-12-31  2026-12-31  5.708099e+09   

d   value_date2    abs_change  pct_change  
0  7.627875e+08  3.239333e+08   73.813398  
1  1.054593e+09  2.263422e+08   27.327733  
2  3.051727e+09  3.777771e+08   14.128055  
3  2.525174e+10  2.435328e+09   10.673584  
4  6.204340e+09  4.962408e+08    8.693625  


In [17]:
# 2) 변화율 표 (예: forecast_date가 NULL, 모든 indicator, 2027-03-31 → 2027-06-30)
df_change = get_indicator_change_rates(
    db_info=db_info,
    forecast_date="2025-11-09",
    indicator="mc_sarima_noexog",
    date1="2025-12-31",
    date2="2026-12",
    table_name="Korea_company_valuation_ver2",
    tolerance_days=10,     # 📌 10일 이내 근사 허용
)
print(df_change.head())


✅ pivot 컬럼 목록:
['indicator', 'ticker', Timestamp('2025-12-31 00:00:00'), Timestamp('2026-11-30 00:00:00')]
📅 실제 사용 날짜: d1_actual=2025-12-31, d2_actual=2026-11-30
✅ 최종 결과 행 수: 45
d         indicator   ticker       date1       date2   value_date1  \
0  mc_sarima_noexog  A298040  2025-12-31  2026-11-30  3.244144e+10   
1  mc_sarima_noexog  A042660  2025-12-31  2026-11-30  5.322986e+10   
2  mc_sarima_noexog  A082740  2025-12-31  2026-11-30  5.747284e+09   
3  mc_sarima_noexog  A006910  2025-12-31  2026-11-30  4.518890e+08   
4  mc_sarima_noexog  A010140  2025-12-31  2026-11-30  3.301526e+10   

d   value_date2    abs_change  pct_change  
0  1.571511e+11  1.247096e+11  384.414582  
1  1.007487e+11  4.751881e+10   89.270972  
2  1.036306e+10  4.615772e+09   80.312230  
3  7.325701e+08  2.806811e+08   62.112859  
4  4.704593e+10  1.403067e+10   42.497535  


In [18]:
df_change

d,indicator,ticker,date1,date2,value_date1,value_date2,abs_change,pct_change
0,mc_sarima_noexog,A298040,2025-12-31,2026-11-30,3.244144e+10,1.571511e+11,1.247096e+11,384.414582
1,mc_sarima_noexog,A042660,2025-12-31,2026-11-30,5.322986e+10,1.007487e+11,4.751881e+10,89.270972
2,mc_sarima_noexog,A082740,2025-12-31,2026-11-30,5.747284e+09,1.036306e+10,4.615772e+09,80.312230
3,mc_sarima_noexog,A006910,2025-12-31,2026-11-30,4.518890e+08,7.325701e+08,2.806811e+08,62.112859
4,mc_sarima_noexog,A010140,2025-12-31,2026-11-30,3.301526e+10,4.704593e+10,1.403067e+10,42.497535
5,mc_sarima_noexog,A000500,2025-12-31,2026-11-30,1.950559e+09,2.664068e+09,7.135092e+08,36.579736
6,mc_sarima_noexog,A071280,2025-12-31,2026-11-30,1.451360e+08,1.884368e+08,4.330079e+07,29.834619
7,mc_sarima_noexog,A131290,2025-12-31,2026-11-30,9.160091e+08,1.112038e+09,1.960289e+08,21.400325
8,mc_sarima_noexog,A103590,2025-12-31,2026-11-30,3.342279e+09,3.925512e+09,5.832333e+08,17.450169
9,mc_sarima_noexog,A042700,2025-12-31,2026-11-30,1.350755e+10,1.576792e+10,2.260371e+09,16.734130


In [12]:

ticker_list = df_change["ticker"].unique().tolist()
df_mck = get_last_market_cap(db_info, "ks_listed_company_daily_marketcap", ticker_list,
                              indicator_exact="시가총액")  # 또는 indicator_like="%market_cap%"
df_mck['market_cap_adj'] = df_mck['last_market_cap']/1000
merged_df = pd.merge(df_change, df_mck[["ticker", "last_market_cap", "market_cap_adj"]], on=["ticker"], how="left")
merged_df['upside_potential'] = (merged_df['value_date2'] -  merged_df['market_cap_adj'])/merged_df['market_cap_adj']

In [14]:
merged_df

,indicator,ticker,date1,date2,value_date1,value_date2,abs_change,pct_change,last_market_cap,market_cap_adj,upside_potential
0,mc_sarima_noexog,A298040,2025-12-31,2026-11-30,3.244144e+10,1.571511e+11,1.247096e+11,384.414582,2.148380e+13,2.148380e+10,6.314863
1,mc_sarima_noexog,A042660,2025-12-31,2026-11-30,5.322986e+10,1.007487e+11,4.751881e+10,89.270972,3.879190e+13,3.879190e+10,1.597157
2,mc_sarima_noexog,A082740,2025-12-31,2026-11-30,5.747284e+09,1.036306e+10,4.615772e+09,80.312230,3.546500e+12,3.546500e+09,1.922051
3,mc_sarima_noexog,A006910,2025-12-31,2026-11-30,4.518890e+08,7.325701e+08,2.806811e+08,62.112859,2.608790e+11,2.608790e+08,1.808084
4,mc_sarima_noexog,A010140,2025-12-31,2026-11-30,3.301526e+10,4.704593e+10,1.403067e+10,42.497535,2.323200e+13,2.323200e+10,1.025049
5,mc_sarima_noexog,A000500,2025-12-31,2026-11-30,1.950559e+09,2.664068e+09,7.135092e+08,36.579736,1.305250e+12,1.305250e+09,1.041040
6,mc_sarima_noexog,A071280,2025-12-31,2026-11-30,1.451360e+08,1.884368e+08,4.330079e+07,29.834619,1.697920e+11,1.697920e+08,0.109810
7,mc_sarima_noexog,A131290,2025-12-31,2026-11-30,9.160091e+08,1.112038e+09,1.960289e+08,21.400325,6.205460e+11,6.205460e+08,0.792032
8,mc_sarima_noexog,A103590,2025-12-31,2026-11-30,3.342279e+09,3.925512e+09,5.832333e+08,17.450169,2.970800e+12,2.970800e+09,0.321365
9,mc_sarima_noexog,A042700,2025-12-31,2026-11-30,1.350755e+10,1.576792e+10,2.260371e+09,16.734130,1.236200e+13,1.236200e+10,0.275515


In [24]:
# 오늘 날짜
today_date = datetime.now().strftime("%Y%m%d")

# 예측 날짜 설정 (필요시 수정)
FORECAST_DATE = "2025-11-12"

# 저장 경로
output_dir = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Results\Korea\Revenue"
output_filename = f"revenue_forecast_total_{today_date}.xlsx"
output_path = os.path.join(output_dir, output_filename)

# 결과를 저장할 빈 리스트
all_results = []

print("=" * 70)
print(f"Revenue 예측 데이터 수집 시작")
print(f"예측 날짜: {FORECAST_DATE}")
print(f"전체 티커 수: {len(ALL_TICKERS)}")
print("=" * 70)

# 티커별로 반복
for idx, ticker_info in enumerate(ALL_TICKERS, 1):
    ticker = ticker_info['ticker']

    print(f"\n[{idx}/{len(ALL_TICKERS)}] 처리 중: {ticker}")

    try:
        # get_series_by_keyword 함수 호출 (이 함수는 이미 정의되어 있다고 가정)
        ex_long = get_series_by_keyword(
            ticker=ticker,
            forecast_date=FORECAST_DATE,
            keyword="rev",
            wide=False
        )

        # 데이터가 없는 경우 건너뛰기
        if ex_long is None or len(ex_long) == 0:
            print(f"  ⚠️  데이터 없음 - 건너뜀")
            continue

        # ex_long을 pivot 테이블로 변환
        ex_pivot = (
            ex_long
            .pivot_table(
                index=["date", "ticker"],
                columns="indicator",
                values="value",
                aggfunc="last"
            )
            .reset_index()
        )

        # 컬럼명 정리
        ex_pivot.columns.name = None

        # 필요한 컬럼만 추출
        available_columns = ['date', 'ticker']
        forecast_columns = ['revenue_ets', 'revenue_prophet', 'revenue_sarima',
                          'revenue_sarima_exog', 'revenue_theta']

        # 실제 존재하는 컬럼만 선택
        for col in forecast_columns:
            if col in ex_pivot.columns:
                available_columns.append(col)

        extracted_df = ex_pivot[available_columns].copy()

        # 결과 리스트에 추가
        all_results.append(extracted_df)

        print(f"  ✓ 성공: {len(extracted_df)} 행 수집")

    except Exception as e:
        print(f"  ✗ 오류 발생: {str(e)}")
        continue

print("\n" + "=" * 70)
print("데이터 수집 완료")
print("=" * 70)

# 모든 결과를 하나의 DataFrame으로 결합
if len(all_results) > 0:
    final_df = pd.concat(all_results, ignore_index=True)

    print(f"\n최종 데이터프레임 크기: {final_df.shape}")
    print(f"  - 총 행 수: {len(final_df):,}")
    print(f"  - 총 열 수: {len(final_df.columns)}")
    print(f"  - 티커 수: {final_df['ticker'].nunique()}")

    # 디렉토리가 없으면 생성
    os.makedirs(output_dir, exist_ok=True)

    # Excel 파일로 저장
    print(f"\n파일 저장 중...")
    final_df.to_excel(output_path, index=False, engine='openpyxl')

    print(f"✓ 저장 완료: {output_path}")

    # 미리보기
    print("\n데이터 미리보기 (처음 5행):")
    print(final_df.head())

else:
    print("\n⚠️  수집된 데이터가 없습니다.")

print("\n" + "=" * 70)
print("작업 완료")
print("=" * 70)


Revenue 예측 데이터 수집 시작
예측 날짜: 2025-11-12
전체 티커 수: 50

[1/50] 처리 중: A005930
  ✓ 성공: 92 행 수집

[2/50] 처리 중: A000660
  ✓ 성공: 92 행 수집

[3/50] 처리 중: A051910
  ✓ 성공: 92 행 수집

[4/50] 처리 중: A035420
  ✓ 성공: 92 행 수집

[5/50] 처리 중: A005380
  ✓ 성공: 92 행 수집

[6/50] 처리 중: A006400
  ✓ 성공: 92 행 수집

[7/50] 처리 중: A035720
  ✓ 성공: 92 행 수집

[8/50] 처리 중: A000270
  ✓ 성공: 92 행 수집

[9/50] 처리 중: A068270
  ✓ 성공: 89 행 수집

[10/50] 처리 중: A207940
  ✓ 성공: 49 행 수집

[11/50] 처리 중: A042700
  ✓ 성공: 89 행 수집

[12/50] 처리 중: A043150
  ✓ 성공: 85 행 수집

[13/50] 처리 중: A131290
  ✓ 성공: 74 행 수집

[14/50] 처리 중: A006910
  ✓ 성공: 92 행 수집

[15/50] 처리 중: A140860
  ✓ 성공: 60 행 수집

[16/50] 처리 중: A009150
  ✓ 성공: 92 행 수집

[17/50] 처리 중: A095610
  ✓ 성공: 79 행 수집

[18/50] 처리 중: A001440
  ✓ 성공: 92 행 수집

[19/50] 처리 중: A000500
  ✓ 성공: 92 행 수집

[20/50] 처리 중: A004000
  ✓ 성공: 92 행 수집

[21/50] 처리 중: A010120
  ✓ 성공: 92 행 수집

[22/50] 처리 중: A044820
  ✓ 성공: 92 행 수집

[23/50] 처리 중: A010140
  ✓ 성공: 92 행 수집

[24/50] 처리 중: A011780
  ✓ 성공: 92 행 수집

[25/50] 처리 중: A033500